# ✈️ Flight Club — Deep Learning Phase
## Notebook 02: Preprocessing for the Deep Learning Model

**Member 1 — Data Pipeline Developer**

**Goal of this notebook:** Turn the validated data into a form that a PyTorch neural network can use.

**Need for preprocessing:**
A neural network only works with numbers in tensors. It cannot handle missing values, it cannot read text such as "JFK",
and it learns badly when features are on very different scales. Preprocessing fixes all three problems.

Steps (each decision comes from the findings of Notebook 01):
1. Load the data and use the existing chronological split
2. Separate numerical features, categorical features and the target
3. Fill missing numerical values (using **training data only**)
4. Standardise numerical features (using **training data only**)
5. Convert categories into integer indexes (category mappings)
6. Handle unseen categories in validation and test
7. Convert everything into PyTorch tensors
8. Create `Dataset` and `DataLoader` objects
9. Save the outputs for the model-training team

**Golden rule: learn from train, apply to all.**
Anything we *learn* from the data (medians, mean, standard deviation, category lists) is learned from the **training set only**.
Validation and test are only *transformed* with those values.
**Reason:** Validation and test stand in for the future. In real life we cannot know future data when we prepare the model.
If we used them, the model would indirectly "see" the future and our results would be too optimistic. This is **data leakage**.

### 1. Import libraries and load the data

**Libraries used:**
- `pandas` / `numpy`: work with tables and arrays
- `torch`: PyTorch, the deep learning library
- `Dataset`, `DataLoader`: PyTorch tools for giving data to the model in batches
- `StandardScaler`: from scikit-learn, used to scale numerical features

**Random seed:** Some steps (like shuffling) are random. Fixing the seed means that we get the **same result every time**
we run the notebook. This makes our work reproducible and easy to check.

**Sorting by date:** It keeps the flights in time order, which matches our time-based approach.

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import os

# Fix the random seed so results are the same every time we run
torch.manual_seed(42)
np.random.seed(42)

DATA_PATH = "flights_feature_engineered_CORRECTED.csv"

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

# Sort by date to keep the time order
df = df.sort_values("date").reset_index(drop=True)

print("Dataset shape:", df.shape)

Dataset shape: (327346, 24)


### 2. Use the existing chronological split

Separate the data into train, validation and test using the `split` column.

In Notebook 01 we verified that this split is in the correct time order with no overlap.
The AML models used the same split, so the DL model is trained and tested on **exactly the same flights**, which makes the comparison fair.

**`.copy()`:** It makes each set a separate table. When we fill missing values in `train_df`, the original `df` is not changed by accident.

In [2]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape, "|", train_df["date"].min().date(), "to", train_df["date"].max().date())
print("Val  :", val_df.shape, "|", val_df["date"].min().date(), "to", val_df["date"].max().date())
print("Test :", test_df.shape, "|", test_df["date"].min().date(), "to", test_df["date"].max().date())

Train: (217727, 24) | 2013-01-01 to 2013-08-31
Val  : (55628, 24) | 2013-09-01 to 2013-10-31
Test : (53991, 24) | 2013-11-01 to 2013-12-31


### 3. Define the features and the target

List the 17 numerical features, the 4 categorical features and the target, the same lists as in Notebook 01 and the AML models.

**Keeping them separate:** They go into the network in **different ways**:
- Numerical features go **directly** into the network (after scaling).
- Categorical features go through **embedding layers** first.
- The target is **not** an input. It is the answer the model must learn to predict.

In [3]:
categorical_features = ["carrier", "origin", "dest", "route"]

numerical_features = [
    "distance",
    "dep_time_sin", "dep_time_cos",
    "weekday_sin", "weekday_cos",
    "month_sin", "month_cos",
    "arr_time_sin", "arr_time_cos",
    "origin_delay_rate_prev7d", "origin_flights_prev7d",
    "carrier_delay_rate_prev7d", "carrier_flights_prev7d",
    "dest_delay_rate_prev7d", "dest_flights_prev7d",
    "route_delay_rate_prev7d", "route_flights_prev7d",
]

target = "delayed"

print("Numerical features  :", len(numerical_features))
print("Categorical features:", len(categorical_features))

Numerical features  : 17
Categorical features: 4


### 4. Handle missing numerical values (training data only)

**Background (from Notebook 01):** The four delay-rate columns are missing only when there were **0 flights in the previous 7 days**,
because the rate cannot be calculated (0 ÷ 0).

Fill the missing values with the **median of the training set**.

**Filling instead of deleting:** These are real flights, and some of them are in the test set. The model must be able to predict them.

**Median instead of mean:** The median is the middle value, so a few very large or very small values do not pull it
up or down the way they pull the mean. It is a safe "typical" value.

**Filling is safe here:** The matching count column (for example `route_flights_prev7d`) is still **0** for these flights.
So the model can still tell that this flight had no history.

**Training median only:** This follows the golden rule. Validation and test are filled with the **training** median, never their own.

In [4]:
# Learn the medians from TRAINING data only
train_medians = train_df[numerical_features].median()

print("Training medians used for filling:")
print(train_medians[["origin_delay_rate_prev7d", "carrier_delay_rate_prev7d",
                     "dest_delay_rate_prev7d", "route_delay_rate_prev7d"]].round(4))

Training medians used for filling:
origin_delay_rate_prev7d     0.2559
carrier_delay_rate_prev7d    0.2508
dest_delay_rate_prev7d       0.2500
route_delay_rate_prev7d      0.2449
dtype: float64


In [5]:
# Use the same training medians for all three sets
train_df[numerical_features] = train_df[numerical_features].fillna(train_medians)
val_df[numerical_features] = val_df[numerical_features].fillna(train_medians)
test_df[numerical_features] = test_df[numerical_features].fillna(train_medians)

print("Missing values after filling:")
print("Train:", train_df[numerical_features].isna().sum().sum())
print("Val  :", val_df[numerical_features].isna().sum().sum())
print("Test :", test_df[numerical_features].isna().sum().sum())

Missing values after filling:
Train: 0
Val  : 0
Test : 0


**Observation:** No missing values are left in any of the three sets.

### 5. Standardise numerical features (training data only)

First, let us **see** the problem: look at the range of each feature **before** scaling.

In [6]:
# Range of each numerical feature BEFORE scaling (training data)
train_df[numerical_features].describe().T[["min", "max", "mean", "std"]].round(2)

,min,max,mean,std
distance,80.00,4983.00,1045.23,735.55
dep_time_sin,-1.00,1.00,-0.16,0.76
dep_time_cos,-1.00,1.00,-0.36,0.52
weekday_sin,-0.97,0.97,0.04,0.70
weekday_cos,-0.90,1.00,-0.00,0.72
month_sin,-0.50,1.00,0.40,0.50
month_cos,-1.00,1.00,-0.14,0.76
arr_time_sin,-1.00,0.99,-0.33,0.59
arr_time_cos,-1.00,1.00,-0.19,0.71
origin_delay_rate_prev7d,0.09,0.52,0.26,0.08


**Observation (from the output above):** The features are on very different scales. The sine/cosine features are between -1 and 1,
and the delay rates are between 0 and 1, but `distance` and the flight-count features go into the **thousands**.

**Problem for a neural network:**
A neural network multiplies each input by a weight and then adjusts the weights using gradients.
If one input is in the thousands and another is below 1, the large input dominates the calculation and its gradients are much larger.
Training becomes unstable and slow, and the small features are almost ignored.

`StandardScaler` changes each feature so that it has **mean = 0** and **standard deviation = 1**:

$$z = \frac{x - \text{mean}}{\text{std}}$$

Now every feature is on a similar scale, so the network can learn from all of them.

**Scaling the sine/cosine features too, even though they are already small:** It keeps every input on exactly the same scale and keeps the code simple.
It does not lose any information. It only moves and stretches the values.

**Fitting on train only:** The mean and std are values **learned** from data. By the golden rule they come from training only.
- `fit_transform(train)`: learns the mean/std from train **and** applies them
- `transform(val/test)`: only **applies** the training mean/std

In [7]:
scaler = StandardScaler()

# fit_transform on train: learns mean/std AND applies them
X_train_num = scaler.fit_transform(train_df[numerical_features])

# transform only on val and test: applies the TRAIN mean/std
X_val_num = scaler.transform(val_df[numerical_features])
X_test_num = scaler.transform(test_df[numerical_features])

print("Train numerical shape:", X_train_num.shape)
print("Val numerical shape  :", X_val_num.shape)
print("Test numerical shape :", X_test_num.shape)

Train numerical shape: (217727, 17)
Val numerical shape  : (55628, 17)
Test numerical shape : (53991, 17)


In [8]:
# Check: AFTER scaling, every training feature should have mean 0 and std 1
check = pd.DataFrame({
    "train_mean": X_train_num.mean(axis=0).round(3),
    "train_std": X_train_num.std(axis=0).round(3),
    "val_mean": X_val_num.mean(axis=0).round(3),
}, index=numerical_features)
check

,train_mean,train_std,val_mean
distance,0.0,1.0,0.000
dep_time_sin,-0.0,1.0,0.008
dep_time_cos,0.0,1.0,-0.040
weekday_sin,-0.0,1.0,0.042
weekday_cos,0.0,1.0,0.033
month_sin,0.0,1.0,-2.664
month_cos,0.0,1.0,-0.140
arr_time_sin,0.0,1.0,-0.012
arr_time_cos,-0.0,1.0,-0.037
origin_delay_rate_prev7d,0.0,1.0,-1.335


**Observation (from the output above):**
- Every **training** feature now has mean 0 and std 1, so the scaling worked.
- **Validation** means are **not 0**, and some are far from 0. For example, `month_sin` is about -2.7 and the delay-rate features are about -1.
  This is **expected and correct**, because validation was scaled with the *training* mean and std, not its own:
  - Validation contains only September–October, so its month values are very different from the January–August training months.
  - Validation has fewer delays (15.1% vs 25.6%, from Notebook 01), so its delay rates are below the training average, which gives negative values.
- If the validation means were exactly 0, that would mean we had (wrongly) fitted the scaler on validation data.

### 6. Create category mappings

**Problem:** A neural network cannot read text such as `"UA"` or `"JFK"`.

**Embeddings instead of one-hot encoding (used by the AML models):**
One-hot encoding makes one new column per category. `route` alone has 218 categories in training, so that would be 218 columns
that are almost all 0. It also treats every category as completely unrelated to every other one.

**Embeddings** (the main idea of our DL model):
An **embedding layer** is a learnable lookup table. It gives every category its own small vector of numbers, and the model
**learns these vectors during training**. Categories that behave in a similar way (for example two airports with similar delay patterns)
can end up with similar vectors. The embedding layer needs each category as an **integer index** (0, 1, 2, ...), so this step creates those indexes.

**How we build the mappings:**
- Built from **training data only** (golden rule).
- **Index 0 is reserved for "unknown"** (categories never seen in training).
- Known categories start from index 1.
- Categories are **sorted** alphabetically first, so the same category always gets the same index every time we run the notebook.

Example: `{"<unknown>": 0, "9E": 1, "AA": 2, "AS": 3, ...}`

**Note:** The index is only an ID, not an amount. The embedding layer uses it to look up a row in its table, so index 12 is not "bigger" than index 2.

In [9]:
category_maps = {}

for col in categorical_features:
    train_categories = sorted(train_df[col].unique())

    mapping = {"<unknown>": 0}
    for index, category in enumerate(train_categories, start=1):
        mapping[category] = index

    category_maps[col] = mapping

# Look at the carrier mapping as an example
category_maps["carrier"]

{'<unknown>': 0,
 '9E': 1,
 'AA': 2,
 'AS': 3,
 'B6': 4,
 'DL': 5,
 'EV': 6,
 'F9': 7,
 'FL': 8,
 'HA': 9,
 'MQ': 10,
 'OO': 11,
 'UA': 12,
 'US': 13,
 'VX': 14,
 'WN': 15,
 'YV': 16}

In [10]:
# Embedding sizes needed by the model team (number of categories + 1 for unknown)
for col in categorical_features:
    print(f"{col:8s}: {len(category_maps[col])} indexes (including unknown = 0)")

carrier : 17 indexes (including unknown = 0)
origin  : 4 indexes (including unknown = 0)
dest    : 103 indexes (including unknown = 0)
route   : 219 indexes (including unknown = 0)


**Use of these numbers for the model team:** When they create an embedding layer, they must tell it how many rows the lookup table has,
for example `nn.Embedding(219, ...)` for route. If the number is too small, the code crashes when it sees a larger index.

### 7. Encode categories and handle unseen categories

Replace each category with its index from the mapping.

**How unseen categories are handled (from Notebook 01: ILM, LEX and some routes):**
1. `.map()` looks up each category in the mapping.
2. If the category is **not** in the mapping (never seen in training), `.map()` gives `NaN`.
3. `.fillna(0)` turns that `NaN` into **0**, the "unknown" index.

**Reason:** Without this, the code would crash on new airports and routes. With it, the model can still predict those flights
using all the other features (distance, time, historical delays, and so on).

In [11]:
def encode_categories(data, category_maps):
    encoded_columns = []
    for col in categorical_features:
        encoded = data[col].map(category_maps[col])  # unseen category -> NaN
        encoded = encoded.fillna(0).astype(int)       # NaN -> 0 (unknown)
        encoded_columns.append(encoded.values)
    return np.column_stack(encoded_columns)


X_train_cat = encode_categories(train_df, category_maps)
X_val_cat = encode_categories(val_df, category_maps)
X_test_cat = encode_categories(test_df, category_maps)

print("Train categorical shape:", X_train_cat.shape)
print("Val categorical shape  :", X_val_cat.shape)
print("Test categorical shape :", X_test_cat.shape)

Train categorical shape: (217727, 4)
Val categorical shape  : (55628, 4)
Test categorical shape : (53991, 4)


In [12]:
# Before vs after: the first 5 training flights
before = train_df[categorical_features].head().reset_index(drop=True)
after = pd.DataFrame(X_train_cat[:5], columns=[c + "_index" for c in categorical_features])
pd.concat([before, after], axis=1)

,carrier,origin,dest,route,carrier_index,origin_index,dest_index,route_index
0,UA,EWR,IAH,EWR_IAH,12,1,44,35
1,UA,EWR,SAN,EWR_SAN,12,1,82,68
2,WN,LGA,STL,LGA_STL,15,3,95,213
3,MQ,EWR,ORD,EWR_ORD,10,1,67,55
4,UA,EWR,DEN,EWR_DEN,12,1,30,22


In [13]:
# How many values were mapped to "unknown" (index 0)?
unknown_counts = pd.DataFrame({
    "train": (X_train_cat == 0).sum(axis=0),
    "val": (X_val_cat == 0).sum(axis=0),
    "test": (X_test_cat == 0).sum(axis=0),
}, index=categorical_features)
unknown_counts

,train,val,test
carrier,0,0,0
origin,0,0,0
dest,0,51,57
route,0,54,61


**Observation (from the output above):**
- Training has **0** unknown values, as expected, because the mappings were built from it.
- The number of unknown `dest` and `route` values in validation and test is **exactly the same** as the unseen counts found in Notebook 01.
  This confirms that every unseen category was mapped to index 0 correctly.

### 8. Prepare the target values

Take the `delayed` column (0 or 1) for each set.

**Printing the delay rate:** It is a quick check that the targets match Notebook 01 (25.6% / 15.1% / 25.0%),
so we know the rows and labels were not mixed up during preprocessing.

In [14]:
y_train = train_df[target].values
y_val = val_df[target].values
y_test = test_df[target].values

print("Train delay rate:", round(y_train.mean(), 4))
print("Val delay rate  :", round(y_val.mean(), 4))
print("Test delay rate :", round(y_test.mean(), 4))

Train delay rate: 0.2561
Val delay rate  : 0.1505
Test delay rate : 0.2499


### 9. Convert to PyTorch tensors

**Tensor:** PyTorch's version of an array. It is like a NumPy array, but PyTorch can use it to compute gradients
and it can run on a GPU. PyTorch models only accept tensors.

**Data types used:**
| Data | Type | Reason |
|---|---|---|
| Numerical features | `float32` | Decimal numbers. `float32` is PyTorch's standard type for weights, and it uses half the memory of `float64` |
| Categorical indexes | `long` (integer) | `nn.Embedding` only accepts whole-number indexes, because they are row numbers in the lookup table |
| Target | `float32` | The binary classification loss (`BCEWithLogitsLoss`) needs the target as a decimal number (0.0 or 1.0) |

In [15]:
X_train_num_t = torch.tensor(X_train_num, dtype=torch.float32)
X_val_num_t = torch.tensor(X_val_num, dtype=torch.float32)
X_test_num_t = torch.tensor(X_test_num, dtype=torch.float32)

X_train_cat_t = torch.tensor(X_train_cat, dtype=torch.long)
X_val_cat_t = torch.tensor(X_val_cat, dtype=torch.long)
X_test_cat_t = torch.tensor(X_test_cat, dtype=torch.long)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

print("Numerical tensor  :", X_train_num_t.shape, X_train_num_t.dtype)
print("Categorical tensor:", X_train_cat_t.shape, X_train_cat_t.dtype)
print("Target tensor     :", y_train_t.shape, y_train_t.dtype)

Numerical tensor  : torch.Size([217727, 17]) torch.float32
Categorical tensor: torch.Size([217727, 4]) torch.int64
Target tensor     : torch.Size([217727]) torch.float32


**Reading the shapes:** `[217727, 17]` means 217,727 flights, each with 17 numerical features.
`[217727, 4]` means 4 category indexes per flight. `[217727]` means one target per flight.
(`torch.int64` is the same type as `long`.)

### 10. Create the Dataset class

A PyTorch `Dataset` tells PyTorch **how to get one flight** from our data. It must have three methods:
- `__init__`: stores the tensors
- `__len__`: returns the number of flights (so PyTorch knows when one pass over the data is finished)
- `__getitem__`: returns the numerical features, category indexes and target for **one** flight

**Own class:** Our model has **two kinds of input** (numerical and categorical) that go into different parts of the network.
Our class returns them **separately**, so the model can send the categories to the embedding layers and the numbers directly to the network.

In [16]:
class FlightDataset(Dataset):
    def __init__(self, x_num, x_cat, y):
        self.x_num = x_num
        self.x_cat = x_cat
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x_num[index], self.x_cat[index], self.y[index]


train_dataset = FlightDataset(X_train_num_t, X_train_cat_t, y_train_t)
val_dataset = FlightDataset(X_val_num_t, X_val_cat_t, y_val_t)
test_dataset = FlightDataset(X_test_num_t, X_test_cat_t, y_test_t)

print("Train dataset size:", len(train_dataset))
print("Val dataset size  :", len(val_dataset))
print("Test dataset size :", len(test_dataset))

Train dataset size: 217727
Val dataset size  : 55628
Test dataset size : 53991


In [17]:
# Look at one flight from the training dataset
x_num, x_cat, y = train_dataset[0]
print("Numerical features:", x_num)
print("Category indexes  :", x_cat)
print("Target            :", y)

Numerical features: tensor([ 0.4823,  1.5139,  1.0682,  1.0642,  0.8743, -0.7981,  1.5061,  1.9502,
        -0.5337, -0.0383, -6.4433, -0.0799, -2.4112, -0.0801, -1.5716, -0.1049,
        -1.3490])
Category indexes  : tensor([12,  1, 44, 35])
Target            : tensor(0.)


### 11. Create DataLoaders

A `DataLoader` takes the Dataset and gives the data to the model in small groups called **batches**.

**Batches instead of all the data at once:**
- Giving all 217,727 flights at once uses a lot of memory.
- With batches, the model updates its weights **after every batch** (213 times per epoch) instead of once per epoch, so it learns faster.

**Batch size 1024:** It is our **chosen starting value**, not a proven best value. It is large enough to make training fast on
tabular data and small enough to fit easily in memory. The model team can change `BATCH_SIZE` during tuning if needed.

**`shuffle=True` for train only:**
- **Train:** shuffling changes the order of flights in every epoch. Without it, every batch would contain flights from the same few days
  (the data is sorted by date), and the model would learn in a biased order.
- **Validation and test:** `shuffle=False` keeps predictions in the same order as the true labels, which is needed to calculate metrics.

**Does shuffling break the time-based split?** No. Shuffling only mixes flights **inside** the training set (January–August).
No validation or test flights enter training, so the split between past and future is still kept.

In [18]:
BATCH_SIZE = 1024

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Number of training batches  :", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches      :", len(test_loader))

Number of training batches  : 213
Number of validation batches: 55
Number of test batches      : 53


In [19]:
# Check one batch
x_num_batch, x_cat_batch, y_batch = next(iter(train_loader))

print("Numerical batch shape  :", x_num_batch.shape)
print("Categorical batch shape:", x_cat_batch.shape)
print("Target batch shape     :", y_batch.shape)

Numerical batch shape  : torch.Size([1024, 17])
Categorical batch shape: torch.Size([1024, 4])
Target batch shape     : torch.Size([1024])


**Observation:** Each batch has 1024 flights, with 17 numerical features and 4 category indexes each.
The number of batches is the number of flights ÷ 1024, rounded up (for example, 217,727 ÷ 1024 gives 213 batches, and the last batch is smaller).

### 12. Final checks

Before handing the data to the model team, we prove with `assert` statements that nothing went wrong.
If any check fails, the notebook stops with an error, so a mistake cannot pass silently.

We check that:
1. Features and targets have the same number of rows (no rows lost or mixed up)
2. There are no missing or infinite values (they would break training)
3. Every category index is smaller than the embedding size (otherwise the embedding layer crashes)
4. The target only contains 0 and 1

In [20]:
# 1. Same number of rows in features and targets
assert len(X_train_num_t) == len(X_train_cat_t) == len(y_train_t)
assert len(X_val_num_t) == len(X_val_cat_t) == len(y_val_t)
assert len(X_test_num_t) == len(X_test_cat_t) == len(y_test_t)

# 2. No missing or infinite values in numerical inputs
assert torch.isfinite(X_train_num_t).all()
assert torch.isfinite(X_val_num_t).all()
assert torch.isfinite(X_test_num_t).all()

# 3. Category indexes are inside the embedding range
for i, col in enumerate(categorical_features):
    assert X_val_cat_t[:, i].max() < len(category_maps[col])
    assert X_test_cat_t[:, i].max() < len(category_maps[col])

# 4. Target only contains 0 and 1
assert set(y_train_t.unique().tolist()) == {0.0, 1.0}

print("All final checks passed!")

All final checks passed!


### 13. Save outputs for the model-training team

Save everything into one file, `dl_outputs/dl_data.pt`, using `torch.save`.

**Reasons:**
- The model team can load the data with one line, without repeating the preprocessing.
- Everyone uses **exactly the same** preprocessed data, so results are consistent and reproducible.
- We also save the **category mappings, medians and scaler values**. To predict a new flight in the future, the same values must be used to prepare it.

In [21]:
os.makedirs("dl_outputs", exist_ok=True)

torch.save({
    "X_train_num": X_train_num_t, "X_train_cat": X_train_cat_t, "y_train": y_train_t,
    "X_val_num": X_val_num_t, "X_val_cat": X_val_cat_t, "y_val": y_val_t,
    "X_test_num": X_test_num_t, "X_test_cat": X_test_cat_t, "y_test": y_test_t,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "category_maps": category_maps,
    "train_medians": train_medians.to_dict(),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_std": scaler.scale_.tolist(),
}, "dl_outputs/dl_data.pt")

print("Saved: dl_outputs/dl_data.pt")

Saved: dl_outputs/dl_data.pt


In [22]:
# How the model team can load the data in the next notebook:
data = torch.load("dl_outputs/dl_data.pt")

print("Loaded keys:", list(data.keys()))
print("Train numerical shape:", data["X_train_num"].shape)

Loaded keys: ['X_train_num', 'X_train_cat', 'y_train', 'X_val_num', 'X_val_cat', 'y_val', 'X_test_num', 'X_test_cat', 'y_test', 'numerical_features', 'categorical_features', 'category_maps', 'train_medians', 'scaler_mean', 'scaler_std']
Train numerical shape: torch.Size([217727, 17])


### 14. Summary

| Step | Action | Reason | Learned from |
|---|---|---|---|
| Missing values | Filled the 4 delay-rate columns with the median | NaN breaks training. Rows are real flights, so we don't delete them | Train only |
| Scaling | `StandardScaler`, so each feature has mean 0 and std 1 | Features ranged from about 1 to thousands, so scaling keeps training stable | Train only |
| Category mappings | Each carrier/origin/dest/route gets an index | Embedding layers need integer indexes | Train only |
| Unseen categories | Mapped to index 0 ("unknown") | New airports/routes in val/test must not crash the model | n/a |
| Tensors | float32 for numbers, long for categories, float32 for the target | Types required by PyTorch layers and the loss | n/a |
| Dataset & DataLoader | Batch size 1024, shuffle on train only | Memory-efficient batches. Shuffle helps learning. Val/test keep their order | n/a |

**Output handed to the model team:**
- 17 standardised numerical features
- 4 categorical index columns (carrier, origin, dest, route) for embedding layers
- Target tensors for train, validation and test
- Category mappings, which give the embedding sizes: carrier 17, origin 4, dest 103, route 219
- Train / validation / test DataLoaders, and the saved file `dl_outputs/dl_data.pt`